In [1]:
import os
import brie
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

print(brie.__version__)

2.3.0


# BRIE2 on GEX

In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy import stats

# Define the confirmed event IDs
targets = {
    'Kras_exon5': 'ENSMUSG00000030265.15.10464',
    'Nf2_exon15': 'ENSMUSG00000009073.17.18773',
    'Slk_exon13': 'ENSMUSG00000025060.16.602',
    'Lsm14b_exon5': 'ENSMUSG00000039108.15.11827',
    'Dgkd_exon29': 'ENSMUSG00000070738.11.10685',
}

# Load both count files
base_dir = "/projects/b1042/GoyalLab/egrody/extractedData/EGL002/singleCell/BRIE2"
wt_count = sc.read_h5ad(f"{base_dir}/WT/brie_count.h5ad")
ko_count = sc.read_h5ad(f"{base_dir}/KO/brie_count.h5ad")

# Compute pseudo-bulk Psi for each gene in each sample
# Psi = sum(isoform1) / (sum(isoform1) + sum(isoform2))
# isoform1 = inclusion isoform (.in), isoform2 = exclusion isoform (.ex)
results = []

for gene_label, event_id in targets.items():
    for sample, adata_c in [('WT', wt_count), ('KO', ko_count)]:
        if event_id in adata_c.var_names:
            idx = adata_c.var_names.get_loc(event_id)
            iso1 = np.array(adata_c.layers['isoform1'][:, idx].todense()).flatten() \
                   if hasattr(adata_c.layers['isoform1'], 'todense') \
                   else adata_c.layers['isoform1'][:, idx].flatten()
            iso2 = np.array(adata_c.layers['isoform2'][:, idx].todense()).flatten() \
                   if hasattr(adata_c.layers['isoform2'], 'todense') \
                   else adata_c.layers['isoform2'][:, idx].flatten()
            
            total_iso1 = iso1.sum()
            total_iso2 = iso2.sum()
            total = total_iso1 + total_iso2
            psi = total_iso1 / total if total > 0 else np.nan
            n_cells = int((iso1 + iso2 > 0).sum())
            
            results.append({
                'gene': gene_label,
                'event_id': event_id,
                'sample': sample,
                'isoform1_counts': int(total_iso1),
                'isoform2_counts': int(total_iso2),
                'total_counts': int(total),
                'n_cells_with_reads': n_cells,
                'pseudobulk_Psi': psi
            })
        else:
            results.append({
                'gene': gene_label,
                'event_id': event_id,
                'sample': sample,
                'isoform1_counts': 0,
                'isoform2_counts': 0,
                'total_counts': 0,
                'n_cells_with_reads': 0,
                'pseudobulk_Psi': np.nan
            })

df = pd.DataFrame(results)
print(df.to_csv(sep='\t', index=False))

gene	event_id	sample	isoform1_counts	isoform2_counts	total_counts	n_cells_with_reads	pseudobulk_Psi
Kras_exon5	ENSMUSG00000030265.15.10464	WT	76	41	117	94	0.6495726704597473
Kras_exon5	ENSMUSG00000030265.15.10464	KO	26	116	142	123	0.18309858441352844
Nf2_exon15	ENSMUSG00000009073.17.18773	WT	3	0	3	2	1.0
Nf2_exon15	ENSMUSG00000009073.17.18773	KO	0	0	0	0	
Slk_exon13	ENSMUSG00000025060.16.602	WT	3	5	8	6	0.375
Slk_exon13	ENSMUSG00000025060.16.602	KO	4	9	13	13	0.3076923191547394
Lsm14b_exon5	ENSMUSG00000039108.15.11827	WT	9	1	10	9	0.8999999761581421
Lsm14b_exon5	ENSMUSG00000039108.15.11827	KO	26	1	27	21	0.9629629850387573
Dgkd_exon29	ENSMUSG00000070738.11.10685	WT	0	1	1	1	0.0
Dgkd_exon29	ENSMUSG00000070738.11.10685	KO	0	0	0	0	



In [19]:
from scipy.stats import fisher_exact
import numpy as np

# Run Fisher's exact test for all genes with data in both conditions
tests = {
    'Kras_exon5':   {'wt_inc': 76, 'wt_exc': 41, 'ko_inc': 26, 'ko_exc': 116},
    'Slk_exon13':   {'wt_inc': 3,  'wt_exc': 5,  'ko_inc': 4,  'ko_exc': 9},
    'Lsm14b_exon5': {'wt_inc': 9,  'wt_exc': 1,  'ko_inc': 26, 'ko_exc': 1},
}

for gene, counts in tests.items():
    table = np.array([[counts['wt_inc'], counts['ko_inc']],
                      [counts['wt_exc'], counts['ko_exc']]])
    odds_ratio, pvalue = fisher_exact(table)
    
    wt_psi = counts['wt_inc'] / (counts['wt_inc'] + counts['wt_exc'])
    ko_psi = counts['ko_inc'] / (counts['ko_inc'] + counts['ko_exc'])
    wt_total = counts['wt_inc'] + counts['wt_exc']
    ko_total = counts['ko_inc'] + counts['ko_exc']
    
    print(f"{gene}")
    print(f"  WT Psi: {wt_psi:.3f} (n={wt_total}), KO Psi: {ko_psi:.3f} (n={ko_total})")
    print(f"  Delta Psi: {ko_psi - wt_psi:.3f}")
    print(f"  Fisher's exact p = {pvalue:.4e}, odds ratio = {odds_ratio:.2f}")
    print()

Kras_exon5
  WT Psi: 0.650 (n=117), KO Psi: 0.183 (n=142)
  Delta Psi: -0.466
  Fisher's exact p = 1.3428e-14, odds ratio = 8.27

Slk_exon13
  WT Psi: 0.375 (n=8), KO Psi: 0.308 (n=13)
  Delta Psi: -0.067
  Fisher's exact p = 1.0000e+00, odds ratio = 1.35

Lsm14b_exon5
  WT Psi: 0.900 (n=10), KO Psi: 0.963 (n=27)
  Delta Psi: 0.063
  Fisher's exact p = 4.7297e-01, odds ratio = 0.35



In [20]:
print(f"{'Gene'}\t{'Sample'}\t{'Psi'}\t{'95% CI low'}\t{'95% CI high'}\t{'n_reads'}")
for gene, counts in tests.items():
    for sample, inc_key, exc_key in [('WT', 'wt_inc', 'wt_exc'), ('KO', 'ko_inc', 'ko_exc')]:
        inc = counts[inc_key]
        total = counts[inc_key] + counts[exc_key]
        ci = psi_ci(inc, total)
        print(f"{gene}\t{sample}\t{inc/total:.3f}\t{ci[0]:.3f}\t{ci[1]:.3f}\t{total}")

Gene	Sample	Psi	95% CI low	95% CI high	n_reads


NameError: name 'psi_ci' is not defined

In [3]:
# Exporting

In [4]:
import scanpy as sc
import numpy as np
import pandas as pd

targets = {
    'Kras_exon5': 'ENSMUSG00000030265.15.10464',
    'Nf2_exon15': 'ENSMUSG00000009073.17.18773',
    'Slk_exon13': 'ENSMUSG00000025060.16.602',
    'Lsm14b_exon5': 'ENSMUSG00000039108.15.11827',
    'Dgkd_exon29': 'ENSMUSG00000070738.11.10685',
}

base_dir = "/projects/b1042/GoyalLab/egrody/extractedData/EGL002/singleCell/BRIE2"
wt_count = sc.read_h5ad(f"{base_dir}/WT/brie_count.h5ad")
ko_count = sc.read_h5ad(f"{base_dir}/KO/brie_count.h5ad")

def extract_per_cell(adata, sample_label, targets):
    """Extract isoform counts per cell for target exons from a BRIE2 count h5ad."""
    rows = []
    for gene_label, event_id in targets.items():
        if event_id not in adata.var_names:
            print(f"WARNING: {event_id} ({gene_label}) not found in {sample_label}")
            continue
        idx = adata.var_names.get_loc(event_id)
        for layer_name in ('isoform1', 'isoform2'):
            col = adata.layers[layer_name][:, idx]
            # sparse matrix columns return (n,1) matrices; dense returns arrays
            if hasattr(col, 'toarray'):
                col = col.toarray().ravel()
            elif hasattr(col, 'todense'):
                col = np.asarray(col.todense()).ravel()
            else:
                col = np.asarray(col).ravel()
            if layer_name == 'isoform1':
                iso1 = col
            else:
                iso2 = col

        total = iso1 + iso2
        psi = np.where(total > 0, iso1 / total, np.nan)

        for i, barcode in enumerate(adata.obs_names):
            rows.append({
                'cell_barcode': barcode,
                'sample': sample_label,
                'gene': gene_label,
                'event_id': event_id,
                'isoform1': int(iso1[i]),
                'isoform2': int(iso2[i]),
                'total': int(total[i]),
                'psi': psi[i],
            })
    return rows

rows_wt = extract_per_cell(wt_count, 'WT', targets)
rows_ko = extract_per_cell(ko_count, 'KO', targets)

df = pd.DataFrame(rows_wt + rows_ko)

# Write combined, then per-sample
df.to_csv(f"{base_dir}/brie2_per_cell_target_exons.csv", index=False)
df[df['sample'] == 'WT'].to_csv(f"{base_dir}/WT/brie2_per_cell_target_exons.csv", index=False)
df[df['sample'] == 'KO'].to_csv(f"{base_dir}/KO/brie2_per_cell_target_exons.csv", index=False)

print(f"Total rows: {len(df)}")
print(f"WT cells: {df[df['sample']=='WT']['cell_barcode'].nunique()}")
print(f"KO cells: {df[df['sample']=='KO']['cell_barcode'].nunique()}")
print(f"Exons: {df['gene'].nunique()}")
print(df.head(10).to_string(index=False))

/tmp/ipykernel_3683009/2407456705.py:40: RuntimeWarning: invalid value encountered in divide
  psi = np.where(total > 0, iso1 / total, np.nan)


Total rows: 78540
WT cells: 4804
KO cells: 10904
Exons: 5
      cell_barcode sample       gene                    event_id  isoform1  isoform2  total  psi
AAACCCAGTTGGGTAG-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACCCATCATAGACC-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACCCATCCACGTCT-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACGAAAGGGAGTTC-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACGAACAGGTGACA-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACGAACATGACTCA-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACGAAGTACCTAAC-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACGAAGTCTAGATC-1     WT Kras_exon5 ENSMUSG00000030265.15.10464         0         0      0  NaN
AAACGAAGTTTCTATC-1     WT Kras_exon5 ENSMUSG00000030265.15.10464     

# BRIE2 on SALVE

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# Define targets: gene_label -> (event_id, [WT_dirs], [KO_dirs])
base_dir = "/projects/b1042/GoyalLab/egrody/extractedData/EGL002/SALVE/test"

targets = {
    'Kras_exon5': {
        'event_id': 'ENSMUSG00000030265.15.10464',
        'WT': ['WT_Kras'],
        'KO': ['KO_Kras'],
    },
    'Dgkd_exon29': {
        'event_id': 'ENSMUSG00000070738.11.10685',
        'WT': ['WT_Dgkd'],
        'KO': ['KO_Dgkd'],
    },
}


def get_counts(dirs, event_id):
    """Sum isoform1 and isoform2 counts across one or more brie_count.h5ad files."""
    total_iso1 = 0
    total_iso2 = 0
    total_cells = 0

    for d in dirs:
        path = f"{base_dir}/{d}/brie_count.h5ad"
        try:
            adata = sc.read_h5ad(path)
        except FileNotFoundError:
            print(f"  WARNING: {path} not found, skipping.")
            continue

        if event_id not in adata.var_names:
            print(f"  WARNING: {event_id} not found in {d}")
            continue

        idx = adata.var_names.get_loc(event_id)

        iso1 = np.array(adata.layers['isoform1'][:, idx].todense()).flatten() \
               if hasattr(adata.layers['isoform1'], 'todense') \
               else adata.layers['isoform1'][:, idx].flatten()
        iso2 = np.array(adata.layers['isoform2'][:, idx].todense()).flatten() \
               if hasattr(adata.layers['isoform2'], 'todense') \
               else adata.layers['isoform2'][:, idx].flatten()

        total_iso1 += iso1.sum()
        total_iso2 += iso2.sum()
        total_cells += int((iso1 + iso2 > 0).sum())

    return int(total_iso1), int(total_iso2), total_cells


# --- Pseudo-bulk Psi ---
results = []
for gene_label, info in targets.items():
    event_id = info['event_id']
    for condition in ['WT', 'KO']:
        iso1, iso2, n_cells = get_counts(info[condition], event_id)
        total = iso1 + iso2
        psi = iso1 / total if total > 0 else np.nan
        results.append({
            'gene': gene_label,
            'event_id': event_id,
            'sample': condition,
            'dirs': ','.join(info[condition]),
            'isoform1_counts': iso1,
            'isoform2_counts': iso2,
            'total_counts': total,
            'n_cells_with_reads': n_cells,
            'pseudobulk_Psi': round(psi, 4) if not np.isnan(psi) else np.nan,
        })

df = pd.DataFrame(results)
print("=== Pseudo-bulk Psi ===")
print(df.to_csv(sep='\t', index=False))

# --- Fisher's exact test ---
print("\n=== Fisher's exact test ===")
print("gene\tWT_Psi\tKO_Psi\tdelta_Psi\tp_value\todds_ratio\tWT_total\tKO_total")

for gene_label in targets:
    wt_row = df[(df['gene'] == gene_label) & (df['sample'] == 'WT')].iloc[0]
    ko_row = df[(df['gene'] == gene_label) & (df['sample'] == 'KO')].iloc[0]

    wt_inc, wt_exc = wt_row['isoform1_counts'], wt_row['isoform2_counts']
    ko_inc, ko_exc = ko_row['isoform1_counts'], ko_row['isoform2_counts']

    wt_total = wt_inc + wt_exc
    ko_total = ko_inc + ko_exc

    if wt_total == 0 or ko_total == 0:
        print(f"{gene_label}\t"
              f"{wt_row['pseudobulk_Psi']}\t{ko_row['pseudobulk_Psi']}\t"
              f"NA\tNA\tNA\t{wt_total}\t{ko_total}")
        continue

    table = np.array([[wt_inc, ko_inc],
                      [wt_exc, ko_exc]])
    odds_ratio, pvalue = fisher_exact(table)

    wt_psi = wt_inc / wt_total
    ko_psi = ko_inc / ko_total
    delta = ko_psi - wt_psi

    print(f"{gene_label}\t{wt_psi:.4f}\t{ko_psi:.4f}\t{delta:.4f}\t"
          f"{pvalue:.4e}\t{odds_ratio:.2f}\t{wt_total}\t{ko_total}")

=== Pseudo-bulk Psi ===
gene	event_id	sample	dirs	isoform1_counts	isoform2_counts	total_counts	n_cells_with_reads	pseudobulk_Psi
Kras_exon5	ENSMUSG00000030265.15.10464	WT	WT_Kras	10281557	0	10281557	2597	1.0
Kras_exon5	ENSMUSG00000030265.15.10464	KO	KO_Kras	1612780	0	1612780	3178	1.0
Dgkd_exon29	ENSMUSG00000070738.11.10685	WT	WT_Dgkd	7768	0	7768	47	1.0
Dgkd_exon29	ENSMUSG00000070738.11.10685	KO	KO_Dgkd	173433	0	173433	1471	1.0


=== Fisher's exact test ===
gene	WT_Psi	KO_Psi	delta_Psi	p_value	odds_ratio	WT_total	KO_total
Kras_exon5	1.0000	1.0000	0.0000	1.0000e+00	nan	10281557	1612780
Dgkd_exon29	1.0000	1.0000	0.0000	1.0000e+00	nan	7768	173433


In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# Define targets: gene_label -> (event_id, [WT_dirs], [KO_dirs])
base_dir = "/projects/b1042/GoyalLab/egrody/extractedData/EGL002/SALVE/BRIE2"

targets = {
    'Kras_exon5': {
        'event_id': 'ENSMUSG00000030265.15.10464',
        'WT': ['WT_Kras'],
        'KO': ['KO_Kras'],
    },
    'Nf2_exon15': {
        'event_id': 'ENSMUSG00000009073.17.18773',
        'WT': ['WT_Nf2'],
        'KO': ['KO_Nf2'],
    },
    'Slk_exon13': {
        'event_id': 'ENSMUSG00000025060.16.602',
        'WT': ['WT_Slkv3', 'WT_Slkv4'],
        'KO': ['KO_Slkv3'],
    },
    'Lsm14b_exon5': {
        'event_id': 'ENSMUSG00000039108.15.11827',
        'WT': ['WT_Lsm14b'],
        'KO': ['KO_Lsm14b'],
    },
    'Dgkd_exon29': {
        'event_id': 'ENSMUSG00000070738.11.10685',
        'WT': ['WT_Dgkd'],
        'KO': ['KO_Dgkd'],
    },
}


def get_counts(dirs, event_id):
    """Sum isoform1 and isoform2 counts across one or more brie_count.h5ad files."""
    total_iso1 = 0
    total_iso2 = 0
    total_cells = 0

    for d in dirs:
        path = f"{base_dir}/{d}/brie_count.h5ad"
        try:
            adata = sc.read_h5ad(path)
        except FileNotFoundError:
            print(f"  WARNING: {path} not found, skipping.")
            continue

        if event_id not in adata.var_names:
            print(f"  WARNING: {event_id} not found in {d}")
            continue

        idx = adata.var_names.get_loc(event_id)

        iso1 = np.array(adata.layers['isoform1'][:, idx].todense()).flatten() \
               if hasattr(adata.layers['isoform1'], 'todense') \
               else adata.layers['isoform1'][:, idx].flatten()
        iso2 = np.array(adata.layers['isoform2'][:, idx].todense()).flatten() \
               if hasattr(adata.layers['isoform2'], 'todense') \
               else adata.layers['isoform2'][:, idx].flatten()

        total_iso1 += iso1.sum()
        total_iso2 += iso2.sum()
        total_cells += int((iso1 + iso2 > 0).sum())

    return int(total_iso1), int(total_iso2), total_cells


# --- Pseudo-bulk Psi ---
results = []
for gene_label, info in targets.items():
    event_id = info['event_id']
    for condition in ['WT', 'KO']:
        iso1, iso2, n_cells = get_counts(info[condition], event_id)
        total = iso1 + iso2
        psi = iso1 / total if total > 0 else np.nan
        results.append({
            'gene': gene_label,
            'event_id': event_id,
            'sample': condition,
            'dirs': ','.join(info[condition]),
            'isoform1_counts': iso1,
            'isoform2_counts': iso2,
            'total_counts': total,
            'n_cells_with_reads': n_cells,
            'pseudobulk_Psi': round(psi, 4) if not np.isnan(psi) else np.nan,
        })

df = pd.DataFrame(results)
print("=== Pseudo-bulk Psi ===")
print(df.to_csv(sep='\t', index=False))

# --- Fisher's exact test ---
print("\n=== Fisher's exact test ===")
print("gene\tWT_Psi\tKO_Psi\tdelta_Psi\tp_value\todds_ratio\tWT_total\tKO_total")

for gene_label in targets:
    wt_row = df[(df['gene'] == gene_label) & (df['sample'] == 'WT')].iloc[0]
    ko_row = df[(df['gene'] == gene_label) & (df['sample'] == 'KO')].iloc[0]

    wt_inc, wt_exc = wt_row['isoform1_counts'], wt_row['isoform2_counts']
    ko_inc, ko_exc = ko_row['isoform1_counts'], ko_row['isoform2_counts']

    wt_total = wt_inc + wt_exc
    ko_total = ko_inc + ko_exc

    if wt_total == 0 or ko_total == 0:
        print(f"{gene_label}\t"
              f"{wt_row['pseudobulk_Psi']}\t{ko_row['pseudobulk_Psi']}\t"
              f"NA\tNA\tNA\t{wt_total}\t{ko_total}")
        continue

    table = np.array([[wt_inc, ko_inc],
                      [wt_exc, ko_exc]])
    odds_ratio, pvalue = fisher_exact(table)

    wt_psi = wt_inc / wt_total
    ko_psi = ko_inc / ko_total
    delta = ko_psi - wt_psi

    print(f"{gene_label}\t{wt_psi:.4f}\t{ko_psi:.4f}\t{delta:.4f}\t"
          f"{pvalue:.4e}\t{odds_ratio:.2f}\t{wt_total}\t{ko_total}")

=== Pseudo-bulk Psi ===
gene	event_id	sample	dirs	isoform1_counts	isoform2_counts	total_counts	n_cells_with_reads	pseudobulk_Psi
Kras_exon5	ENSMUSG00000030265.15.10464	WT	WT_Kras	0	0	0	0	
Kras_exon5	ENSMUSG00000030265.15.10464	KO	KO_Kras	99892	0	99892	69	1.0
Nf2_exon15	ENSMUSG00000009073.17.18773	WT	WT_Nf2	9372	13	9385	16	0.9986
Nf2_exon15	ENSMUSG00000009073.17.18773	KO	KO_Nf2	1049	18	1067	7	0.9831
Slk_exon13	ENSMUSG00000025060.16.602	WT	WT_Slkv3,WT_Slkv4	31742	45265	77007	67	0.4122
Slk_exon13	ENSMUSG00000025060.16.602	KO	KO_Slkv3	5640	0	5640	78	1.0
Lsm14b_exon5	ENSMUSG00000039108.15.11827	WT	WT_Lsm14b	807198	0	807198	32	1.0
Lsm14b_exon5	ENSMUSG00000039108.15.11827	KO	KO_Lsm14b	166530	0	166530	80	1.0
Dgkd_exon29	ENSMUSG00000070738.11.10685	WT	WT_Dgkd	2823	0	2823	6	1.0
Dgkd_exon29	ENSMUSG00000070738.11.10685	KO	KO_Dgkd	29	0	29	15	1.0


=== Fisher's exact test ===
gene	WT_Psi	KO_Psi	delta_Psi	p_value	odds_ratio	WT_total	KO_total
Kras_exon5	nan	1.0	NA	NA	NA	0	99892
Nf2_exon15	0.9986	0.983

# Checking for genes

In [13]:
# Load the count file to check
adata_count = sc.read_h5ad(dat_dir + "/brie_count.h5ad")

for symbol, ensid in gene_map.items():
    matches = adata_count.var[adata_count.var['GeneID'].str.startswith(ensid)]
    if len(matches) > 0:
        for eid in matches.index:
            idx = adata_count.var_names.get_loc(eid)
            total = adata_count.layers['isoform1'][:, idx].sum() + \
                    adata_count.layers['isoform2'][:, idx].sum()
            n_cells = (adata_count.layers['isoform1'][:, idx] + 
                       adata_count.layers['isoform2'][:, idx] > 0).sum()
            print(f"{symbol} ({eid}): total_counts={total}, cells_with_counts={n_cells}")
    else:
        print(f"{symbol} ({ensid}): NOT in brie_count either")

Kras (ENSMUSG00000030265.15.10464): total_counts=117.0, cells_with_counts=94
Kras (ENSMUSG00000030265.15.10465): total_counts=74.0, cells_with_counts=65
Kras (ENSMUSG00000030265.15.10466): total_counts=144.0, cells_with_counts=108
Kras (ENSMUSG00000030265.15.10467): total_counts=5.0, cells_with_counts=3
Nf2 (ENSMUSG00000009073.17.18773): total_counts=3.0, cells_with_counts=2
Nf2 (ENSMUSG00000009073.17.18774): total_counts=2.0, cells_with_counts=2
Nf2 (ENSMUSG00000009073.17.18775): total_counts=0.0, cells_with_counts=0
Nf2 (ENSMUSG00000009073.17.18776): total_counts=0.0, cells_with_counts=0
Slk (ENSMUSG00000025060.16.602): total_counts=8.0, cells_with_counts=6
Slk (ENSMUSG00000025060.16.603): total_counts=3.0, cells_with_counts=3
Lsm14b (ENSMUSG00000039108.15.11826): total_counts=5.0, cells_with_counts=3
Lsm14b (ENSMUSG00000039108.15.11827): total_counts=10.0, cells_with_counts=9
Dgkd (ENSMUSG00000070738.11.10684): total_counts=0.0, cells_with_counts=0
Dgkd (ENSMUSG00000070738.11.10685)

In [15]:
import subprocess

for eid in ['ENSMUSG00000030265.15.10464', 'ENSMUSG00000030265.15.10465', 
            'ENSMUSG00000030265.15.10466',
            'ENSMUSG00000009073.17.18773',
            'ENSMUSG00000025060.16.602',
            'ENSMUSG00000039108.15.11826', 'ENSMUSG00000039108.15.11827',
            'ENSMUSG00000070738.11.10685']:
    result = subprocess.run(
        ['grep', eid, '/projects/b1042/GoyalLab/egrody/genomes/SE_events_mm39.gff3'],
        capture_output=True, text=True
    )
    # Print only the gene and skipped exon lines
    lines = result.stdout.strip().split('\n')
    gene_line = [l for l in lines if '\tgene\t' in l]
    skipped = [l for l in lines if '.in.2' in l]  # middle exon = skipped exon
    print(f"\n--- {eid} ---")
    if gene_line:
        print(gene_line[0])
    if skipped:
        print(skipped[0])


--- ENSMUSG00000030265.15.10464 ---
chr6	rMATS	gene	145165971	145177980	.	-	.	ID=ENSMUSG00000030265.15.10464;gene_id=ENSMUSG00000030265.15.10464;gene_name=ENSMUSG00000030265.15
chr6	rMATS	exon	145170800	145170923	.	-	.	ID=ENSMUSG00000030265.15.10464.in.2;Parent=ENSMUSG00000030265.15.10464.in

--- ENSMUSG00000030265.15.10465 ---
chr6	rMATS	gene	145162425	145192542	.	-	.	ID=ENSMUSG00000030265.15.10465;gene_id=ENSMUSG00000030265.15.10465;gene_name=ENSMUSG00000030265.15
chr6	rMATS	exon	145177821	145177980	.	-	.	ID=ENSMUSG00000030265.15.10465.in.2;Parent=ENSMUSG00000030265.15.10465.in

--- ENSMUSG00000030265.15.10466 ---
chr6	rMATS	gene	145177821	145192542	.	-	.	ID=ENSMUSG00000030265.15.10466;gene_id=ENSMUSG00000030265.15.10466;gene_name=ENSMUSG00000030265.15
chr6	rMATS	exon	145179974	145180152	.	-	.	ID=ENSMUSG00000030265.15.10466.in.2;Parent=ENSMUSG00000030265.15.10466.in

--- ENSMUSG00000009073.17.18773 ---
chr11	rMATS	gene	4717665	4732352	.	-	.	ID=ENSMUSG00000009073.17.18773;gene_id=ENS